# Neoracer V1 — Async Core Test

This notebook walks through every sensor, control surface, and driver feature exposed by the v2 student library and the `neoracer_ros2_driver` backend. Run each cell in order. The car must be powered on with the teleop stack already running:

```bash
ros2 launch neoracer_ros2_driver teleop.launch.py
```

Each section runs for a fixed window (defaults to 10 s) and reports live counters. The final "Summary" cell aggregates pass/fail for every subsystem.

The entire system is based on the `racecar_neo_ros2_driver`, and treats upstream updates as authority. The Neoracer software is designed to be compatible with either vehicle, regardless of hardware capability.

## 1. Initialize Racecar

In [ ]:
import sys, os, time, io

import numpy as np
import cv2 as cv
from IPython.display import display
import ipywidgets as widgets

import racecar_core
import racecar_utils as rc_utils

In [ ]:
def to_jpeg_bytes(bgr_image, resize=(320, 240)):
    """Convert a BGR numpy image to JPEG bytes for widget display."""
    if resize:
        bgr_image = cv.resize(bgr_image, resize)
    _, buf = cv.imencode('.jpg', bgr_image, [cv.IMWRITE_JPEG_QUALITY, 70])
    return buf.tobytes()

In [ ]:
# Create the racecar in real mode and start the async ROS2 executor
rc = racecar_core.create_racecar(isSimulation=False)
rc.go_async()
print('Racecar initialized, async executor running.')
print('Waiting 3 seconds for topics to connect...')
time.sleep(3)

## 2. Forward Camera Stream (10 seconds)

Subscribes to `/camera/color` (`sensor_msgs/Image`). On the NeoRacer this is the
USB webcam's native MJPG buffer passed through unchanged, so `encoding` is
`jpeg` rather than the `bgr8` a RealSense publishes; the library decodes it with
`cv2.imdecode`. The reference platform puts its D435i colour stream on the same
topic, so lab code does not need to know which camera it is talking to.


In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=320, height=240)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = rc.camera.get_color_image_async()
    if color_image is not None:
        frame_count += 1
        img_widget.value = to_jpeg_bytes(color_image)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = f'Forward camera | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | {fps:.1f} FPS'
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'Forward camera complete: {frame_count} frames in {elapsed:.1f}s = {rate:.1f} FPS'
forward_received = rc.camera.get_color_image_async() is not None

## 3. LIDAR Stream (10 seconds)

Subscribes to `/scan` (`sensor_msgs/LaserScan`). On the physical car `rc.lidar.get_num_samples()` returns 1080 (RPLIDAR with `angle_compensate=true`). The sim returns 720. Always use `get_num_samples()` instead of hard-coding either length.

In [ ]:
DURATION = 10
MAX_RANGE_CM = 1000
img_widget = widgets.Image(format='jpeg', width=320, height=320)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    scan = rc.lidar.get_samples_async()
    if scan is not None and len(scan) > 0:
        frame_count += 1
        radius = 160
        image = np.zeros((2 * radius, 2 * radius, 3), np.uint8)
        n = len(scan)
        for i in range(n):
            d = scan[i]
            if 0 < d < MAX_RANGE_CM:
                angle = 2 * np.pi * i / n
                length = radius * d / MAX_RANGE_CM
                r = int(radius - length * np.cos(angle))
                c = int(radius + length * np.sin(angle))
                if 0 <= r < 2 * radius and 0 <= c < 2 * radius:
                    image[r, c, 2] = 255
        cv.circle(image, (radius, radius), 3, (0, 255, 0), -1)
        img_widget.value = to_jpeg_bytes(image, resize=None)
        forward = rc_utils.get_lidar_average_distance(scan, 0) if hasattr(rc_utils, 'get_lidar_average_distance') else float(scan[0])
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = (
            f'LIDAR | Samples: {n} | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | '
            f'{fps:.1f} Hz | Forward: {forward:.0f} cm'
        )
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'LIDAR complete: {frame_count} scans in {elapsed:.1f}s = {rate:.1f} Hz'

## 4. IMU / Physics Data

Subscribes to `/imu/fused` (`sensor_msgs/Imu`) and `/mag`
(`sensor_msgs/MagneticField`). On the reference platform `/imu/fused` is
`imu_fusion_node` blending the D435i and LSM9DS1; on the NeoRacer it is the
single ESP32 IMU published under the same name. The IMU frame is x=front,
y=right, z=up; gravity should read ~9.81 m/s2 along whichever axis points up.


In [ ]:
for i in range(5):
    accel = rc.physics.get_linear_acceleration()
    gyro = rc.physics.get_angular_velocity()
    mag = rc.physics.get_magnetic_field()

    print(f'--- Sample {i+1} ---')
    print(f'  Accel: ({accel[0]:7.2f}, {accel[1]:7.2f}, {accel[2]:7.2f}) m/s^2')
    print(f'  Gyro:  ({gyro[0]:7.3f}, {gyro[1]:7.3f}, {gyro[2]:7.3f}) rad/s')
    print(f'  Mag:   ({mag[0]:+.3e}, {mag[1]:+.3e}, {mag[2]:+.3e}) T')
    time.sleep(0.2)

accel_mag = float(np.linalg.norm(accel))
print(f'\nAccel magnitude: {accel_mag:.2f} m/s^2 (expected ~9.81 when stationary)')
if 8.0 < accel_mag < 12.0:
    print('PASS: IMU acceleration magnitude looks correct')
else:
    print('WARN: Acceleration magnitude outside expected range — check calibration')

## 5. Inference (10 seconds)

Subscribes to `/edgetpu/inference` (`vision_msgs/Detection2DArray`) through
`rc.vision`. The topic name comes from the reference platform, which runs a
Coral EdgeTPU; the NeoRacer serves the same message type from a YOLO model on
the Orin's integrated GPU. Overlays bounding boxes on the live camera frame.

The inference node is part of the default teleop stack. If this section reports
nothing, check that it is running (`racecar status`) and that
`racecar setup ml` has been run on this car; the node exits at startup when
ultralytics is not importable.


In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=640, height=480)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
total_detections = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = rc.camera.get_color_image_async()
    detections = rc.vision.get_detections_async()
    if color_image is not None:
        frame_count += 1
        annotated = color_image.copy()
        for det in detections:
            cx, cy, w, h = det.bbox
            x1, y1 = int(cx - w / 2), int(cy - h / 2)
            x2, y2 = int(cx + w / 2), int(cy + h / 2)
            cv.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv.putText(
                annotated, f'{det.class_id} {det.score:.0%}', (x1, y1 - 8),
                cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
            )
        total_detections += len(detections)
        img_widget.value = to_jpeg_bytes(annotated, resize=None)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = (
            f'YOLO | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | '
            f'{fps:.1f} FPS | Detections: {len(detections)} | Total: {total_detections}'
        )
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'EdgeTPU complete: {frame_count} frames in {elapsed:.1f}s = {rate:.1f} FPS | Total detections: {total_detections}'

## 6. Drive Stack — mux observation (30 seconds)

Subscribes to `/mux_out` (`ackermann_msgs/AckermannDriveStamped`) so we can see exactly what the mux is forwarding to the throttle/pwm chain. The driver mux selects between `/gamepad_drive` and the student's `/drive` based on the LB/RB bumpers:

- **No bumper** → IDLE (zero)
- **LB held** → GAMEPAD (raw joystick → mux)
- **RB held** → AUTONOMY (student `/drive` → mux). This cell publishes a slow square-wave on `rc.drive` while RB is held, so you can confirm the mux is forwarding student commands.

In [ ]:
import rclpy as ros2
from ackermann_msgs.msg import AckermannDriveStamped
from rclpy.qos import QoSProfile, QoSReliabilityPolicy, QoSDurabilityPolicy

mux_node = ros2.create_node('mux_test_sub')
mux_qos = QoSProfile(depth=1)
mux_qos.reliability = QoSReliabilityPolicy.BEST_EFFORT
mux_qos.durability = QoSDurabilityPolicy.VOLATILE
mux_latest = [AckermannDriveStamped()]

def _mux_cb(msg):
    mux_latest[0] = msg

mux_node.create_subscription(AckermannDriveStamped, '/mux_out', _mux_cb, mux_qos)
rc._RacecarReal__executor.add_node(mux_node)

DURATION = 30

# Drive command amplitude. The signal goes through three multiplicative
# scalers before reaching the ESC, so a small value here disappears into the
# ESC deadband:
#   - student-side:  msg.drive.speed = command * STUDENT_MAX_SPEED  (drive_real)
#   - throttle_node: out.speed       = in.speed  * max_speed_forward (~0.5)
#   - pwm_node:      pwm = center + sign * speed * magnitude (3000 us swing)
# Net forward PWM offset = STUDENT_MAX_SPEED * STUDENT_AMPLITUDE * 0.5 * 3000.
# An ESC needs roughly >= +/-300 us off center before the motor starts to
# turn, so the inputs below give ~+/-450 us — comfortably past the deadband
# but still gentle for a benchtop test. Drop these if the car has tires off
# the ground and you want to see motor spin without rolling.
STUDENT_MAX_SPEED = 0.5
STUDENT_AMPLITUDE = 0.6
STUDENT_STEERING  = 0.4

timer_label = widgets.Label(value='Starting...')
bumper_label = widgets.Label(value='Bumpers: ...')
mode_label = widgets.Label(value='Mux mode: ...')
output_label = widgets.Label(value='Mux output: ...')
student_label = widgets.Label(value='Student /drive: ...')

display(widgets.VBox([timer_label, bumper_label, mode_label, student_label, output_label]))

rc.drive.set_max_speed(STUDENT_MAX_SPEED)
rc.drive.stop()

t_start = time.monotonic()
last_mux_stamp = 0.0
while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Drive / Mux | {elapsed:.1f}s / {DURATION}s'

    lb = rc.controller.is_down(rc.controller.Button.LB)
    rb = rc.controller.is_down(rc.controller.Button.RB)
    bumper_label.value = f'Bumpers:   LB={"HELD" if lb else "---"}  RB={"HELD" if rb else "---"}'

    if rb and not lb:
        phase = (elapsed % 3.0) < 1.5
        speed = STUDENT_AMPLITUDE if phase else -STUDENT_AMPLITUDE
        angle = STUDENT_STEERING  if phase else -STUDENT_STEERING
        rc.drive.set_speed_angle(speed, angle)
        mode_label.value = 'Mux mode:  AUTONOMY (RB) — student /drive forwarded'
        # Show both the raw command and what hits /drive after the student
        # library's max_speed scaling, so the wheels-not-moving case is
        # unambiguous (raw 0.6 vs published 0.3 vs mux-out 0.3 vs PWM offset).
        published_speed = speed * STUDENT_MAX_SPEED
        student_label.value = (
            f'Student /drive: raw=({speed:+.2f}, {angle:+.2f})  '
            f'after max_speed={STUDENT_MAX_SPEED}: speed={published_speed:+.3f}'
        )
    elif lb and not rb:
        rc.drive.stop()
        mode_label.value = 'Mux mode:  GAMEPAD (LB) — /gamepad_drive forwarded'
        student_label.value = 'Student /drive: stopped (mux is on the gamepad source)'
    else:
        rc.drive.stop()
        mode_label.value = 'Mux mode:  IDLE — zero out'
        student_label.value = 'Student /drive: stopped'

    m = mux_latest[0]
    mux_age = elapsed - last_mux_stamp if last_mux_stamp else 0.0
    if m.header.stamp.sec or m.header.stamp.nanosec:
        last_mux_stamp = elapsed
    output_label.value = (
        f'Mux out:   speed={m.drive.speed:+.3f}  steering_angle={m.drive.steering_angle:+.3f}  '
        f'(stamp_sec={m.header.stamp.sec})'
    )
    time.sleep(0.05)

rc.drive.stop()
timer_label.value = f'Drive / Mux test complete ({DURATION}s)'
rc._RacecarReal__executor.remove_node(mux_node)
mux_node.destroy_node()

## 7. Battery Power - Voltage

`rc.physics.get_battery_voltage()` returns the pack voltage in volts, published
on `/battery/voltage` by the `controller` node from the ESP32's `b` frame at
about 0.5 Hz, so expect roughly four samples in the window below.

`get_battery_current()` is not exercised here: the NeoRacer's OSRbot base has no
current shunt, so it raises `NotImplementedError`. The reference platform reads
an INA226 on the NEO-PIT board and does support it.


In [ ]:
DURATION = 8

timer_label = widgets.Label(value='Starting...')
volt_label = widgets.Label(value='Voltage: ...')
display(widgets.VBox([timer_label, volt_label]))

v_min, v_max = 1e9, -1e9
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    v = rc.physics.get_battery_voltage()
    v_min, v_max = min(v_min, v), max(v_max, v)
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Battery power | {elapsed:.1f}s / {DURATION}s'
    volt_label.value = f'Voltage: {v:6.2f} V   (min {v_min:.2f}, max {v_max:.2f})'
    time.sleep(0.1)

v = rc.physics.get_battery_voltage()
power_ok = 5.0 < v < 13.0
print(f'Final: {v:.2f} V')
print('PASS: voltage in [5, 13] V' if power_ok
      else 'WARN: voltage outside [5, 13] V - check the pack')


## 8. FlySky RC Controller

Reads the 8-channel FlySky iA6B receiver through `rc.physics.get_rc_channels()`, which returns eight values normalized to `[-1, 1]` (0 at center / no signal). Turn the transmitter ON and move both sticks and the switches.

Channel map (verified on hardware):

| idx | control | idx | control |
|-----|---------|-----|---------|
| 0 | right stick X | 4 | switch A |
| 1 | right stick Y | 5 | switch B |
| 2 | left stick Y (throttle) | 6 | switch C |
| 3 | left stick X | 7 | switch D |

In [ ]:
DURATION = 10
CH_NAMES = ['0 right-X', '1 right-Y', '2 left-Y (throttle)', '3 left-X',
            '4 switch-A', '5 switch-B', '6 switch-C', '7 switch-D']

timer_label = widgets.Label(value='Starting... (turn the transmitter ON, move sticks + switches)')
ch_labels = [widgets.Label(value=f'{n}: 0.00') for n in CH_NAMES]
display(widgets.VBox([timer_label] + ch_labels))

mins = [9.0] * 8
maxs = [-9.0] * 8
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    ch = rc.physics.get_rc_channels()               # 8 values, each normalized to [-1, 1]
    for i in range(8):
        v = float(ch[i])
        mins[i] = min(mins[i], v)
        maxs[i] = max(maxs[i], v)
        ch_labels[i].value = f'{CH_NAMES[i]}: {v:+.2f}   (min {mins[i]:+.2f}, max {maxs[i]:+.2f})'
    elapsed = time.monotonic() - t_start
    timer_label.value = f'FlySky RC | {elapsed:.1f}s / {DURATION}s'
    time.sleep(0.05)

moved = sum(1 for i in range(8) if (maxs[i] - mins[i]) > 0.3)
rc_ok = moved > 0
print(f'{moved} channel(s) swung more than 0.3 during the window.')
print('PASS: transmitter detected and channels responding' if rc_ok
      else 'WARN: no channel moved — is the transmitter on and bound?')

## 9. Encoder Speed

`rc.physics.get_encoder_speed()` returns the car's forward speed in m/s, derived from the NEO-PIT hall encoder (the Teensy applies the gear ratios and wheel circumference). The library does not expose raw motor rpm, so this cell also derives an approximate **wheel** rpm from the wheel circumference for reference. Spin a wheel by hand or drive the car to see it change.

In [ ]:
DURATION = 10
WHEEL_DIAMETER_M = 0.072                    # NEO-PIT wheel (firmware CAR_SPECS)
WHEEL_CIRC_M = np.pi * WHEEL_DIAMETER_M

timer_label = widgets.Label(value='Starting... (spin a wheel by hand or drive)')
speed_label = widgets.Label(value='Speed: ...')
rpm_label = widgets.Label(value='Wheel rpm: ...')
display(widgets.VBox([timer_label, speed_label, rpm_label]))

max_abs = 0.0
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    speed = rc.physics.get_encoder_speed()          # m/s, gear ratios applied on the Teensy
    wheel_rpm = (speed / WHEEL_CIRC_M) * 60.0        # derived reference (library exposes m/s, not raw rpm)
    max_abs = max(max_abs, abs(speed))
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Encoder | {elapsed:.1f}s / {DURATION}s'
    speed_label.value = f'Speed: {speed:+.3f} m/s   (max |v| {max_abs:.3f})'
    rpm_label.value = f'Wheel rpm: {wheel_rpm:+.1f}'
    time.sleep(0.05)

encoder_ok = max_abs > 0.0
print(f'Peak speed seen: {max_abs:.3f} m/s')
print('PASS: encoder responded to motion' if encoder_ok
      else 'INFO: no motion detected — spin a wheel during the window to confirm the encoder')

## 10. Dot-matrix Display

The NeoRacer panel is an 8x8 module behind a serial firmware that renders ASCII
and nothing else, so `/dotmatrix/text` (`std_msgs/String`) is the whole
interface and `rc.display.show_text()` is the only call that reaches it.

`set_matrix()`, `get_matrix()`, and `set_matrix_intensity()` raise
`NotImplementedError` here. The reference platform drives a 24x8 MAX7219 chain
and accepts raw frames on `/dotmatrix/pixels`; this car has no equivalent.

The panel holds the last frame written, so the section ends on
`rc.display.clear()`, which returns it to the idle frame `led_matrix_node`
writes on startup. A new frame is applied only when the current scroll cycle
ends, so writes sent closer together than one cycle can be swallowed.


In [ ]:
# show_text() is the whole dot-matrix interface on this car; the panel firmware
# takes ASCII only. set_matrix()/set_matrix_intensity() raise here by design.
rc.display.show_text('NEORACER')
print('Sent "NEORACER" to /dotmatrix/text. Holding 5 s...')
time.sleep(5)

# Anything wider than the panel scrolls automatically in firmware.
rc.display.show_text('TOPIC CONTRACT OK')
print('Sent a longer string; the firmware scrolls it.')
time.sleep(10)

# One write to end on; a second before the scroll cycle ends can be swallowed.
rc.display.clear()
dotmatrix_ok = True
print('Cleared to the idle frame.')

# Confirm the unsupported calls fail loudly rather than silently doing nothing.
for name, call in (('set_matrix', lambda: rc.display.set_matrix(rc.display.new_matrix())),
                   ('set_matrix_intensity', lambda: rc.display.set_matrix_intensity(0.5))):
    try:
        call()
        print(f'UNEXPECTED: {name} did not raise')
    except NotImplementedError:
        print(f'expected: {name} raises NotImplementedError on this platform')


## 11. Telemetry — record and visualize

`rc.telemetry.declare_variables(...)` opens a timestamped CSV under `labs/logs/`; `record(...)` appends a row; `visualize()` writes a sibling PNG. We log accel magnitude and a synthetic sine for 5 seconds and then plot.

In [ ]:
rc.telemetry.declare_variables('accel_mag', 'sine')

DURATION = 5
t_start = time.monotonic()
n = 0
while time.monotonic() - t_start < DURATION:
    t = time.monotonic() - t_start
    a = rc.physics.get_linear_acceleration()
    rc.telemetry.record(float(np.linalg.norm(a)), float(np.sin(2 * np.pi * 0.5 * t)))
    n += 1
    time.sleep(0.05)

rc.telemetry.visualize()
print(f'Logged {n} telemetry samples in {DURATION}s.')
print(f'CSV: {rc.telemetry._LOG_FILE_NAME}')
print(f'PNG: {rc.telemetry._PLOT_FILE_NAME}')

## 12. Diagnostics

`/diagnostics` (`diagnostic_msgs/DiagnosticArray`) carries inference timing,
frame counts, and device health. On the NeoRacer it comes from `inference_node`
(the reference platform's `edgetpu_node` fills the same role), plus any other
diagnostics-emitting driver node. `rc.telemetry.get_diagnostics()` returns a
dict keyed by status name.

Expect this to be empty unless the inference node is running.


In [ ]:
diag = rc.telemetry.get_diagnostics()

if diag:
    for name, info in diag.items():
        level = info['level']
        if isinstance(level, (bytes, bytearray)):
            level = int.from_bytes(level, byteorder='big')
        level_str = {0: 'OK', 1: 'WARN', 2: 'ERROR', 3: 'STALE'}.get(level, f'LVL:{level}')
        print(f'[{level_str}] {name}: {info["message"]}')
        for k, v in info.items():
            if k not in ('level', 'message', 'hardware_id'):
                print(f'    {k}: {v}')
        print()
else:
    print('No diagnostics received yet. Confirm edgetpu_enable:=true on the teleop launch.')

## 13. Summary

In [ ]:
color = rc.camera.get_color_image_async()
scan = rc.lidar.get_samples_async()
accel = rc.physics.get_linear_acceleration()
diagnostics = rc.telemetry.get_diagnostics()
detections = rc.vision.get_detections_async()
voltage = rc.physics.get_battery_voltage()
rc_channels = rc.physics.get_rc_channels()

results = {
    'Forward camera (/camera/color)': color is not None,
    'LIDAR (/scan)': scan is not None and len(scan) > 0,
    'IMU (/imu/fused, /mag)': bool(np.any(accel != 0)),
    'Controller (/joy)': rc.controller.get_joystick(rc.controller.Joystick.LEFT) is not None,
    'Drive publisher (/drive)': True,
    'Mux observation (/mux_out)': True,
    'Battery voltage (/battery/voltage)': 5.0 < voltage < 13.0,
    'FlySky RC (/rc/channels)': rc_channels is not None and len(rc_channels) == 8,
    'Encoder (/encoder/speed)': rc.physics.get_encoder_speed() is not None,
    'Dot-matrix (/dotmatrix/text)': True,
    'Telemetry (CSV + PNG + /diagnostics)': rc.telemetry._LOG_FILE_NAME is not None,
    'Vision (/edgetpu/inference)': len(detections) >= 0,
    'Diagnostics (/diagnostics)': len(diagnostics) > 0,
}

# Present in the racecar_neo API, no hardware behind them on this car. Each
# raises NotImplementedError naming the missing part; that is the contract, not
# a failure, so they are reported separately from the pass/fail table.
unsupported = {
    'Depth stream (/camera/depth)': 'no depth sensor; forward camera is monocular',
    'Battery current (/battery/current)': 'no current shunt on the OSRbot base',
    'LED strip (/led/pixels)': 'no addressable strip',
    'Dot-matrix pixels (/dotmatrix/pixels)': 'panel firmware takes text only',
}

print('=' * 64)
print('  NeoRacer - Async Core Test Results')
print('=' * 64)
all_pass = True
for name, ok in results.items():
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_pass = False
    print(f'  [{status}] {name}')
print('-' * 64)
print('  Not available on this platform (raises NotImplementedError):')
for name, why in unsupported.items():
    print(f'  [ -- ] {name}')
    print(f'         {why}')
print('=' * 64)
print(f'  Overall: {"ALL PASS" if all_pass else "SOME FAILURES - check above"}')
print('=' * 64)
